In [1]:
import os
import json
from moviepy import (
    AudioFileClip,
    CompositeVideoClip,
    ColorClip,
    TextClip,
    VideoFileClip,
    concatenate_videoclips,
    vfx
)
import re
import random
from collections import defaultdict

In [2]:
def demoji(text):
    emoji_pattern = re.compile("["
                            u"\U0001F600-\U0001F64F"  # emoticons
                            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                            u"\U0001F680-\U0001F6FF"  # transport & map symbols
                            u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                            "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [ ]:
INPUT_FILE = r"./export/result.json"
INPUT_DIR = r"./export/"          # папка с исходными видео
OUTPUT_FILE = r"./result/output.mp4"    # итоговый файл
W, H = 1920, 1080
VIDEO_FONT_PATH = "./static/Roboto-Regular.ttf"
TRANSITION_DURATION = 0.6   # плавный переход между видео

TITLE_FONT_PATH = "./static/Roboto-Bold.ttf"
TITLE_TEXT = "АНТИДОБРОКЕК #10"
INTRO_MUSIC = "./static/intro.mp3"         # музыка для вступления
TITLE_DURATION = 30.0  # секунды (30-сек рэп трек)

BPM = 150
BEAT = 60 / BPM  # 0.4s

In [4]:
export_dict = {}
with open(INPUT_FILE, 'r') as input_file:
    export_dict = json.load(input_file)

In [5]:
statistics = defaultdict(int)

In [6]:
videos = []
messages = random.sample(export_dict["messages"], k=len(export_dict["messages"]))
for message in messages:
    if message.get("media_type", "unknown")!= "video_file":
        continue

    title = ""
    skip = False
    for text_entity in message["text_entities"]:
        if text_entity["type"] in ("hashtag") and text_entity["text"] == "#dobrokek":
            skip = True
            break
        elif text_entity["type"] not in ("plain", "code"):
            continue
        elif text_entity["type"] == "code":
            statistics[text_entity["text"]] += 1

        title += text_entity["text"].replace("\n\n", " ").strip()+" "
    if skip:
        continue
    video = {
        "filepath": os.path.join(INPUT_DIR, message["file"]),
        "title": demoji(title).strip(),
    }

    videos.append(video)


In [7]:
import math

# --- Beat flashes (без with_opacity — кодируем яркость в цвет) ---
def make_beat_flashes(duration, beat, w=W, h=H):
    flashes = []
    i = 0
    t = 0.0
    while t < duration - 0.05:
        # Даунбит 0.4 → (102,102,102), остальные 0.12 → (31,31,31)
        brightness = 102 if i % 4 == 0 else 31
        flash = (
            ColorClip((w, h), color=(brightness, brightness, brightness))
            .with_duration(0.04)
            .with_start(round(t, 4))
        )
        flashes.append(flash)
        i += 1
        t = round(t + beat, 4)
    return flashes

# --- Shake ---
def shake(t, amp_x=8, amp_y=5):
    x = math.sin(t * 47.3) * amp_x
    y = math.cos(t * 31.7) * amp_y
    return x, y

# --- Eased slide ---
def ease_out(progress):
    return 1 - (1 - progress) ** 3

CENTER_Y = H // 2 - 120
SLIDE_DUR = 0.6

ANTI_FINAL_Y     = CENTER_Y - 140
DOBROKEK_FINAL_Y = CENTER_Y + 80

# АНТИ летит сверху — стартует с -219 (1px виден, moviepy не падает)
def pos_anti(t):
    progress = ease_out(min(1.0, t / SLIDE_DUR))
    y = -219 + (ANTI_FINAL_Y - (-219)) * progress
    sx, sy = shake(t) if t > SLIDE_DUR else (0, 0)
    return (int(sx), int(y + sy))

# ДОБРОКЕК летит снизу — стартует с H-1 (1px виден)
def pos_dobrokek(t):
    progress = ease_out(min(1.0, t / SLIDE_DUR))
    y = (H - 1) + (DOBROKEK_FINAL_Y - (H - 1)) * progress
    sx, sy = shake(t) if t > SLIDE_DUR else (0, 0)
    return (int(sx), int(y + sy))

# RGB Split + shake
def pos_white(t):
    sx, sy = shake(t)
    return (int(sx), int(CENTER_Y + sy))

def pos_red(t):
    sx, sy = shake(t)
    return (int(sx - 10), int(CENTER_Y + sy - 2))

def pos_blue(t):
    sx, sy = shake(t)
    return (int(sx + 10), int(CENTER_Y + sy + 2))

def pos_number(t):
    sx, sy = shake(t, amp_x=5, amp_y=3)
    return (int(sx), int(CENTER_Y + 220 + sy))

# --- Фон ---
title_bg = ColorClip((W, H), color=(0, 0, 0), duration=TITLE_DURATION)

beat_flashes = make_beat_flashes(TITLE_DURATION, BEAT)

# --- Фаза 1 (0–10s): АНТИ сверху ---
anti = TextClip(
    font=TITLE_FONT_PATH,
    text="АНТИ",
    color="#FF2222",
    font_size=160,
    size=(W, 220),
    text_align="center",
    method='caption',
).with_start(BEAT).with_duration(10.0 - BEAT) \
 .with_effects([vfx.FadeOut(0.05)]) \
 .with_position(pos_anti)

# --- Фаза 2 (4–10s): ДОБРОКЕК снизу ---
dobrokek = TextClip(
    font=TITLE_FONT_PATH,
    text="ДОБРОКЕК",
    color="white",
    font_size=120,
    size=(W, 180),
    text_align="center",
    method='caption',
).with_start(4.0).with_duration(6.0) \
 .with_effects([vfx.FadeOut(0.05)]) \
 .with_position(pos_dobrokek)

# --- Фаза 3 (10–30s): АНТИДОБРОКЕК с RGB Split + Shake ---
main_dur = TITLE_DURATION - 10.0

main_white = TextClip(
    font=TITLE_FONT_PATH,
    text="АНТИДОБРОКЕК",
    color="white",
    font_size=130,
    size=(W, 200),
    text_align="center",
    method='caption',
).with_start(10.0).with_duration(main_dur) \
 .with_effects([vfx.FadeIn(0.04)]) \
 .with_position(pos_white)

# RGB red — цвет pre-multiplied (без with_opacity)
main_red = TextClip(
    font=TITLE_FONT_PATH,
    text="АНТИДОБРОКЕК",
    color="#C81E1E",
    font_size=130,
    size=(W, 200),
    text_align="center",
    method='caption',
).with_start(10.0).with_duration(main_dur) \
 .with_position(pos_red)

# RGB blue — цвет pre-multiplied
main_blue = TextClip(
    font=TITLE_FONT_PATH,
    text="АНТИДОБРОКЕК",
    color="#1E50DC",
    font_size=130,
    size=(W, 200),
    text_align="center",
    method='caption',
).with_start(10.0).with_duration(main_dur) \
 .with_position(pos_blue)

# --- #10 золотом (22–30s) ---
number_text = TextClip(
    font=TITLE_FONT_PATH,
    text="#10",
    color="#FFD700",
    font_size=200,
    size=(W, 280),
    text_align="center",
    method='caption',
).with_start(22.0).with_duration(TITLE_DURATION - 22.0) \
 .with_effects([vfx.FadeIn(0.1)]) \
 .with_position(pos_number)

# --- Ударные вспышки (без with_opacity — цвет = opacity*255) ---
dobrokek_flash = ColorClip((W, H), color=(128, 128, 128)).with_duration(0.06).with_start(4.0)
reveal_flash   = ColorClip((W, H), color=(217, 217, 217)).with_duration(0.06).with_start(10.0)
gold_flash     = ColorClip((W, H), color=(128, 100,   0)).with_duration(0.06).with_start(22.0)

# --- Собираем ---
title_clip = CompositeVideoClip(
    [title_bg] +
    beat_flashes +
    [dobrokek_flash, reveal_flash, gold_flash,
     anti, dobrokek,
     main_red, main_blue, main_white,
     number_text]
)

# Музыка
audio_tracks = []
if os.path.exists(INTRO_MUSIC):
    print("Intro music added")
    intro_music = AudioFileClip(INTRO_MUSIC).with_volume_scaled(1.0).subclipped(0, TITLE_DURATION)
    title_clip = title_clip.with_audio(intro_music)
    audio_tracks.append(intro_music)

Intro music added


In [8]:
video_tracks = [title_clip.with_start(0)]
current_time = TITLE_DURATION

In [9]:
clips = [title_clip]

In [24]:
FIRST_CLIP_PATH = r"./export/first_clip.MP4"

In [25]:
first_clip = VideoFileClip(FIRST_CLIP_PATH)

# Масштабирование
first_clip_resized = first_clip.resized(height=H)
if first_clip_resized.w > W:
    first_clip_resized = first_clip_resized.resized(width=W)

# Подложка
first_clip_bg = ColorClip((W, H), color=(0, 0, 0), duration=first_clip.duration)


# Собираем видео
first_clip_composite = CompositeVideoClip([
    first_clip_bg,
    first_clip_resized.with_position("center"),
]).with_audio(first_clip.audio)


clips.append(first_clip_composite)

In [10]:
for video in videos[:2]:
    try:
        clip = VideoFileClip(video["filepath"])

        # Масштабирование
        clip_resized = clip.resized(height=H)
        if clip_resized.w > W:
            clip_resized = clip_resized.resized(width=W)

        # Подложка
        bg = ColorClip((W, H), color=(0, 0, 0), duration=clip.duration)

        # Подпись к видео — в углу справа
        caption_w = (W-clip_resized.w)//2
        caption = TextClip(
            font=VIDEO_FONT_PATH,
            size=(caption_w, H),
            text=video["title"],
            color="white",
            font_size=20,
        ).with_duration(clip.duration).with_position(("right", "top"))

        # Собираем видео
        composite = CompositeVideoClip([
            bg,
            clip_resized.with_position("center"),
            caption
        ]).with_audio(clip.audio)


        clips.append(composite)
    except:
        continue

In [ ]:
LAST_CLIP_PATH = r"./export/last_clip.mp4"

In [ ]:
last_clip = VideoFileClip(LAST_CLIP_PATH)

# Масштабирование
last_clip_resized = last_clip.resized(height=H)
if last_clip_resized.w > W:
    last_clip_resized = last_clip_resized.resized(width=W)

# Подложка
last_clip_bg = ColorClip((W, H), color=(0, 0, 0), duration=last_clip.duration)


# Собираем видео
last_clip_composite = CompositeVideoClip([
    last_clip_bg,
    last_clip_resized.with_position("center"),
]).with_audio(last_clip.audio)


clips.append(last_clip_composite)

In [ ]:
sorted_donors = sorted(statistics.items(), key=lambda x: x[1], reverse=True)[:5]
sorted_donors = [
    {
        "name": "Саша 1"
        "amount": "5"
    }
]
# 2. Теперь собираем строку
donors_list_text = "\n".join([
    f"{i+1}. {name} – {amount}" 
    for i, (name, amount) in enumerate(sorted_donors)
])
donors_list_text += "\n\nВСЕМ СПАСИБО ЗА ВИДЕО!"

In [12]:
END_DURATION = 4.65
OUTRO_MUSIC = "./static/outro.mp3" 

DONOR_FONT_SIZES = [72, 58, 48, 40, 34]
DONOR_COLORS     = ["#FFD700", "#C0C0C0", "#CD7F32", "#AAAAAA", "#AAAAAA"]
DONOR_MEDALS     = ["🥇", "🥈", "🥉", " 4.", " 5."]

# Фон
end_bg = ColorClip((W, H), color=(0, 0, 0), duration=END_DURATION)

# Заголовок
end_title = TextClip(
    font=TITLE_FONT_PATH,
    text=TITLE_TEXT,
    color="white",
    font_size=64,
    size=(W, int(H * 0.15)),
    text_align="center",
    method='caption',
).with_duration(END_DURATION).with_position(("center", 40))

# Подзаголовок
end_label = TextClip(
    font=TITLE_FONT_PATH,
    text="ТОП АНТИДОБРОКЕКЕРОВ ВЫПУСКА",
    color="#CCCCCC",
    font_size=40,
    size=(W, int(H * 0.1)),
    text_align="center",
    method='caption',
).with_duration(END_DURATION).with_position(("center", 155))

# Отступ где начинаются донатеры
DONORS_START_Y = 290
donor_clips = []

for i, (name, amount) in enumerate(sorted_donors):
    font_size = DONOR_FONT_SIZES[i] if i < len(DONOR_FONT_SIZES) else 30
    color     = DONOR_COLORS[i]     if i < len(DONOR_COLORS)     else "#AAAAAA"
    label     = f"{i+1}. {name} – {amount}"

    # Высота строки = font_size * 1.5
    y = DONORS_START_Y + sum(
        int((DONOR_FONT_SIZES[j] if j < len(DONOR_FONT_SIZES) else 30) * 1.6)
        for j in range(i)
    )

    clip = TextClip(
        font=TITLE_FONT_PATH,
        text=label,
        color=color,
        font_size=font_size,
        size=(W, int(font_size * 1.6)),
        text_align="center",
        method='caption',
    ).with_duration(END_DURATION - i * 0.3) \
     .with_start(i * 0.3) \
     .with_effects([vfx.FadeIn(0.2)]) \
     .with_position(("center", y))

    donor_clips.append(clip)

# Итоговая надпись
thanks_y = DONORS_START_Y + sum(
    int((DONOR_FONT_SIZES[j] if j < len(DONOR_FONT_SIZES) else 30) * 1.6)
    for j in range(len(sorted_donors))
) + 20

thanks_clip = TextClip(
    font=TITLE_FONT_PATH,
    text="ВСЕМ СПАСИБО ЗА ВИДЕО!",
    color="white",
    font_size=44,
    size=(W, 70),
    text_align="center",
    method='caption',
).with_duration(END_DURATION - len(sorted_donors) * 0.3) \
 .with_start(len(sorted_donors) * 0.3) \
 .with_effects([vfx.FadeIn(0.3)]) \
 .with_position(("center", thanks_y))

end_clip = CompositeVideoClip([end_bg, end_title, end_label] + donor_clips + [thanks_clip])

if os.path.exists(OUTRO_MUSIC):
    outro_music = AudioFileClip(OUTRO_MUSIC).with_volume_scaled(0.2).subclipped(0, END_DURATION)
    end_clip = end_clip.with_audio(outro_music)

In [29]:
clips.append(end_clip)

In [ ]:
import math

def make_ad_transition(duration=TRANSITION_DURATION, w=W, h=H):
    FONT = TITLE_FONT_PATH
    FS = 260
    BOX_W, BOX_H = 300, 320

    A_START = 0.05
    A_DUR   = duration - A_START - 0.10

    D_START = 0.25
    D_DUR   = duration - D_START - 0.10
    D_SLIDE = 0.20  # длина ease-out скольжения

    # Позиции верхнего-левого угла боксов
    # А: центр в (w//2 - 130, h//2 - 70)
    AX = w // 2 - 130 - BOX_W // 2
    AY = h // 2 - 70  - BOX_H // 2

    # Д: финальный центр в (w//2 + 130, h//2 + 70)
    DX_F = w // 2 + 130 - BOX_W // 2
    DY_F = h // 2 + 70  - BOX_H // 2

    # Д стартует правее и ниже финальной позиции (скользит снизу-справа)
    DX_S = DX_F + 110
    DY_S = DY_F + 140

    def shake(t, ax=6, ay=4):
        return math.sin(t * 47.3) * ax, math.cos(t * 31.7) * ay

    def ease_out(p):
        return 1 - (1 - p) ** 3

    def pos_a(t):
        sx, sy = shake(t)
        return (int(AX + sx), int(AY + sy))

    def pos_a_red(t):
        sx, sy = shake(t)
        return (int(AX - 10 + sx), int(AY - 2 + sy))

    def pos_a_blue(t):
        sx, sy = shake(t)
        return (int(AX + 10 + sx), int(AY + 2 + sy))

    def pos_d(t):
        p = ease_out(min(1.0, t / D_SLIDE))
        x = DX_S + (DX_F - DX_S) * p
        y = DY_S + (DY_F - DY_S) * p
        sx, sy = shake(t) if t > D_SLIDE else (0, 0)
        return (int(x + sx), int(y + sy))

    def pos_d_red(t):
        px, py = pos_d(t)
        return (px - 10, py - 2)

    def pos_d_blue(t):
        px, py = pos_d(t)
        return (px + 10, py + 2)

    bg = ColorClip((w, h), color=(0, 0, 0), duration=duration)

    def make_letter(char, color, start, dur, pos_fn, fade_in=None):
        clip = TextClip(
            font=FONT, text=char, color=color, font_size=FS,
            size=(BOX_W, BOX_H), text_align="center", method='caption',
        ).with_start(start).with_duration(dur).with_position(pos_fn)
        if fade_in:
            clip = clip.with_effects([vfx.FadeIn(fade_in)])
        return clip

    a_white = make_letter("А", "white",   A_START, A_DUR, pos_a,      fade_in=0.10)
    a_red   = make_letter("А", "#C81E1E", A_START, A_DUR, pos_a_red)
    a_blue  = make_letter("А", "#1E50DC", A_START, A_DUR, pos_a_blue)

    d_white = make_letter("Д", "white",   D_START, D_DUR, pos_d,      fade_in=0.07)
    d_red   = make_letter("Д", "#C81E1E", D_START, D_DUR, pos_d_red)
    d_blue  = make_letter("Д", "#1E50DC", D_START, D_DUR, pos_d_blue)

    return CompositeVideoClip(
        [bg, a_red, a_blue, a_white, d_red, d_blue, d_white]
    ).with_effects([vfx.FadeIn(0.10), vfx.FadeOut(0.15)])

transition = make_ad_transition()

In [31]:
final = concatenate_videoclips(
    clips,
    method="chain",
    transition=transition
)

final.write_videofile(
    OUTPUT_FILE,
    fps=30,
    threads=12,
    codec="h264_nvenc",
    ffmpeg_params=[
        "-preset", "p4",       # nvenc preset: p1 (быстро) … p7 (качественно)
        "-cq", "23",           # качество (18=лучше, 28=хуже)
        "-b:v", "0",           # отключаем битрейт, используем cq
        "-rc", "vbr",
    ],
    audio_codec="aac",
)

MoviePy - Building video ./result/output.mp4.
MoviePy - Writing audio in outputTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video ./result/output.mp4



frame_index:   4%|▎         | 2844/76755 [07:40<3:04:13,  6.69it/s, now=None]c:\Users\AlexS\miniconda3\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:190: UserWarning: In file ./export/first_clip.MP4, 1216032 bytes wanted but 0 bytes read at frame index 1926 (out of a total 1926 frames), at time 64.26/64.27 sec. Using the last valid frame instead.
  warnings.warn(
frame_index:   5%|▍         | 3784/76755 [09:38<2:54:50,  6.96it/s, now=None]c:\Users\AlexS\miniconda3\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:190: UserWarning: In file ./export/video_files/reels_20260322_200852.mp4, 1952640 bytes wanted but 0 bytes read at frame index 450 (out of a total 454 frames), at time 15.00/15.16 sec. Using the last valid frame instead.
  warnings.warn(
frame_index:   5%|▍         | 3785/76755 [09:38<3:02:40,  6.66it/s, now=None]c:\Users\AlexS\miniconda3\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:190: UserWarning: In file ./export/video_files/reels_20260322_200852.mp4, 195264

MoviePy - Done !
MoviePy - video ready ./result/output.mp4


In [32]:
# Быстрый превью — первые 15 секунд в половину разрешения
# Запускай эту ячейку вместо финального рендера когда тестируешь

final.subclipped(0, 15).resized(0.5).write_videofile(
    "./result/preview.mp4",
    fps=30,
    codec="h264_nvenc",
    ffmpeg_params=["-preset", "p1", "-cq", "28", "-b:v", "0", "-rc", "vbr"],
    audio_codec="aac",
)

MoviePy - Building video ./result/preview.mp4.
MoviePy - Writing audio in previewTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video ./result/preview.mp4



MoviePy - Done !
MoviePy - video ready ./result/preview.mp4
